# Baseline B: Markov Model / Naive Forecasting

Here we establish two baseline models for forecasting the Federal Funds Target Rate decisions (`higher`, `same`, `lower`) at the **next FOMC meeting**:
1. **Naive Baseline**: "Predict whatever happened at the previous meeting" ($\hat{y}_t = y_{t-1}$)
2. **1st-order Markov Transition**: Calculate the transition probability matrix $P(y_t | y_{t-1})$ and predict the most likely next action.

To accurately estimate real-world performance where we only have past information, we will evaluate these models using two methods:
- **Simple Train/Test Split**: Train on 2009–2018, Test on 2019–2026.
- **Walk-Forward Validation (Rolling Window)**: For each new meeting day, we train on all the meetings that occurred prior to it, make the prediction for the upcoming meeting, and then roll the window forward.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load Data and Extract Valid FOMC Meeting Observations
raw_meetings = [
    '2009-01-28', '2009-03-18', '2009-04-29', '2009-06-24', '2009-08-12', '2009-09-23', '2009-11-04', '2009-12-16', 
    '2010-01-27', '2010-03-16', '2010-04-28', '2010-06-23', '2010-08-10', '2010-09-21', '2010-11-03', '2010-12-14', 
    '2011-01-26', '2011-03-15', '2011-04-27', '2011-06-22', '2011-08-09', '2011-09-21', '2011-11-02', '2011-12-13', 
    '2012-01-25', '2012-03-13', '2012-04-25', '2012-06-20', '2012-07-31', '2012-09-13', '2012-10-24', '2012-12-12', 
    '2013-01-30', '2013-03-20', '2013-05-01', '2013-06-19', '2013-07-31', '2013-09-18', '2013-10-30', '2013-12-18', 
    '2014-01-29', '2014-03-19', '2014-04-30', '2014-06-18', '2014-07-30', '2014-09-17', '2014-10-29', '2014-12-17', 
    '2015-01-28', '2015-03-18', '2015-04-29', '2015-06-17', '2015-07-29', '2015-09-17', '2015-10-28', '2015-12-16', 
    '2016-01-27', '2016-03-16', '2016-04-27', '2016-06-15', '2016-07-27', '2016-09-21', '2016-11-02', '2016-12-14', 
    '2017-02-01', '2017-03-15', '2017-05-03', '2017-06-14', '2017-07-26', '2017-09-20', '2017-11-01', '2017-12-13', 
    '2018-01-31', '2018-03-21', '2018-05-02', '2018-06-13', '2018-08-01', '2018-09-26', '2018-11-08', '2018-12-19', 
    '2019-01-30', '2019-03-19', '2019-05-01', '2019-06-19', '2019-07-31', '2019-09-18', '2019-10-04', '2019-10-30', '2019-12-11', 
    '2020-01-29', '2020-03-03', '2020-03-15', '2020-03-23', '2020-04-29', '2020-06-10', '2020-07-29', '2020-09-16', '2020-11-05', '2020-12-16', 
    '2021-01-27', '2021-03-17', '2021-04-28', '2021-06-16', '2021-07-28', '2021-09-22', '2021-11-03', '2021-12-15', 
    '2022-01-26', '2022-03-16', '2022-05-04', '2022-06-15', '2022-07-27', '2022-09-21', '2022-11-02', '2022-12-14', 
    '2023-02-01', '2023-03-22', '2023-05-03', '2023-06-14', '2023-07-26', '2023-09-20', '2023-11-01', '2023-12-13', 
    '2024-01-31', '2024-03-20', '2024-05-01', '2024-06-12', '2024-07-31', '2024-09-18', '2024-11-07', '2024-12-18', 
    '2025-01-28', '2025-03-19', '2025-05-07', '2025-06-18', '2025-07-30', '2025-09-17', '2025-10-29', '2025-12-10', 
    '2026-01-28'
]

df_meetings = pd.DataFrame({'meeting_date': pd.to_datetime(raw_meetings)}).drop_duplicates().sort_values('meeting_date').reset_index(drop=True)
df_meetings['post_meeting_date'] = df_meetings['meeting_date'] + pd.Timedelta(days=1)

df = pd.read_csv('Federal Funds Target Range Upper Limit.csv')
df['DFEDTARU'] = pd.to_numeric(df['DFEDTARU'], errors='coerce')
df['observation_date'] = pd.to_datetime(df['observation_date'])
df = df.dropna(subset=['DFEDTARU']).sort_values('observation_date').reset_index(drop=True)

df_decisions = pd.merge(df_meetings, df[['observation_date', 'DFEDTARU']], left_on='post_meeting_date', right_on='observation_date', how='left')
df_decisions = df_decisions[['meeting_date', 'DFEDTARU']].rename(columns={'DFEDTARU': 'target_rate'})

def get_action_label(diff_val):
    if pd.isna(diff_val):
        return None
    elif diff_val > 0:
        return 'higher'
    elif diff_val < 0:
        return 'lower'
    else:
        return 'same'

df_decisions['previous_rate'] = df_decisions['target_rate'].shift(1)
df_decisions['label'] = (df_decisions['target_rate'] - df_decisions['previous_rate']).apply(get_action_label)

# Remove the first period (no preceding meeting to compare to)
df_model = df_decisions.dropna(subset=['label']).copy().reset_index(drop=True)

# The core feature for prediction is the previous meeting's decision
df_model['y_t_minus_1'] = df_model['label'].shift(1)
df_model = df_model.dropna(subset=['y_t_minus_1']).copy().reset_index(drop=True)

print("Total meetings available for modeling:", len(df_model))
display(df_model.head())

## Method 1: Simple Train / Test Split

**Train Set**: 2009 to 2018 (Learn the probabilities of transitions)
**Test Set**: 2019 to Present (Evaluate accuracy)

In [ ]:
train_df = df_model[(df_model['meeting_date'].dt.year >= 2009) & (df_model['meeting_date'].dt.year <= 2018)].copy()
test_df = df_model[df_model['meeting_date'].dt.year >= 2019].copy()

print(f"Training set size: {len(train_df)} meetings")
print(f"Test set size: {len(test_df)} meetings")

# --- Naive Baseline --- #
test_df['naive_pred'] = test_df['y_t_minus_1']
acc_naive_split = (test_df['naive_pred'] == test_df['label']).mean()

# --- Markov Model --- #
# Train: Calculate transition probability matrix based only on the Train Set
transition_counts = pd.crosstab(train_df['y_t_minus_1'], train_df['label'])
transition_probs = transition_counts.div(transition_counts.sum(axis=1), axis=0)

print("\n--- Markov Training: Transition Probability Matrix P(y_t | y_t-1) ---")
# We format to string explicitly to avoid jinja2 requirements
display(transition_probs.applymap(lambda x: f'{x:.2%}'))

# Decision strategy: which action has the highest probability given the previous action
markov_strategy = transition_probs.idxmax(axis=1)

# Fallback logic in case a previous state exists in the test set that was never seen in training
def markov_predict(y_prev, strategy):
    if y_prev in strategy.index:
        return strategy[y_prev]
    return 'same'

test_df['markov_pred'] = test_df['y_t_minus_1'].apply(lambda x: markov_predict(x, markov_strategy))
acc_markov_split = (test_df['markov_pred'] == test_df['label']).mean()

print(f"\n[Simple Split] Naive Baseline Accuracy: {acc_naive_split:.2%}")
print(f"[Simple Split] Markov Baseline Accuracy: {acc_markov_split:.2%}")

## Method 2: Walk-Forward Validation (Rolling Forecast)

For the test set period (2019 onwards), instead of training once on 2009-2018 and stopping, we evaluate each meeting by training a new Markov transition matrix using **all data available up to the day before the meeting**.
This perfectly simulates a real-world forecasting environment.

In [ ]:
# The index where 2019 begins
start_idx = df_model[df_model['meeting_date'].dt.year == 2019].index[0]

wf_naive_predictions = []
wf_markov_predictions = []
actual_decisions = []

for i in range(start_idx, len(df_model)):
    # Train window: all meetings from index 0 strictly up to (but not including) the current meeting
    train_history = df_model.iloc[:i]
    current_meeting = df_model.iloc[i]
    
    actual_decisions.append(current_meeting['label'])
    
    # 1. Naive prediction
    wf_naive_predictions.append(current_meeting['y_t_minus_1'])
    
    # 2. Markov Prediction
    # Recalculate transition probabilities on the available history
    counts = pd.crosstab(train_history['y_t_minus_1'], train_history['label'])
    probs = counts.div(counts.sum(axis=1), axis=0)
    strategy = probs.idxmax(axis=1)
    
    prev_state = current_meeting['y_t_minus_1']
    if prev_state in strategy.index:
        pred = strategy[prev_state]
    else:
        pred = 'same' # safe fallback if prev_state is completely unseen
        
    wf_markov_predictions.append(pred)

wf_df = pd.DataFrame({
    'actual': actual_decisions,
    'naive_pred': wf_naive_predictions,
    'markov_pred': wf_markov_predictions
})

acc_naive_wf = (wf_df['naive_pred'] == wf_df['actual']).mean()
acc_markov_wf = (wf_df['markov_pred'] == wf_df['actual']).mean()

print(f"\n[Walk-Forward] Naive Baseline Accuracy: {acc_naive_wf:.2%}")
print(f"[Walk-Forward] Markov Baseline Accuracy: {acc_markov_wf:.2%}")

# Show a breakdown of the Walk-Forward prediction performance (Confusion Matrix for Markov)
print("\n--- Walk-Forward Markov Model Confusion Matrix ---")
display(pd.crosstab(wf_df['actual'], wf_df['markov_pred'], 
            rownames=['Actual Decision'], colnames=['Predicted Decision'], margins=True))